---
## 6. Predicción Multi-Step: t+30 y t+60 Minutos

### Fundamento matemático

Dado el estado del sistema en el instante t, la probabilidad de estar en el estado k
en el instante t+H es:

$$P(Z_{t+H} = k \mid \text{observaciones hasta } t) = [\pi_t \cdot A^H]_k$$

Donde:
- $\pi_t$ es el vector de probabilidades posteriores en t (del algoritmo forward)
- $A$ es la matriz de transición
- $A^H$ es A multiplicada por sí misma H veces

Multiplicar $\pi_t$ por A un paso hacia adelante da la distribución en t+1.
Repetir H veces da la distribución en t+H.

La **concentración esperada** en t+H es la expectativa sobre los estados:

$$E[X_{t+H}] = \sum_k P(Z_{t+H}=k) \cdot \mu_k$$

Donde $\mu_k$ es la media de emisiones del estado k (desnormalizada).

**Importante:** A medida que H crece, la distribución converge a la distribución
estacionaria del modelo. A t+60 min, la incertidumbre es máxima.

# Modelacion de Calidad del Aire con Hidden Markov Model (HMM)
## Zona Metropolitana de Guadalajara â€” Sensores D (Zapopan)

Este notebook implementa un **Hidden Markov Model (HMM)** para modelar la calidad del aire
como un proceso latente. La hipotesis central es que el estado real de la atmosfera â€”limpia,
afectada por trafico, fotoquimica activa, episodio de contaminacionâ€” es una **variable oculta**
que nunca se observa directamente, sino que se infiere de las mediciones de los sensores.

### Diferencia con el approach de regresion (modelacion.ipynb)
El notebook anterior predia el AQI directamente como regresion supervisada. Este notebook
trata la calidad del aire como un **sistema dinamico con estados discretos latentes**:

- Los estados ocultos Z(t) âˆˆ {1, ..., K} representan regimenes atmosfericos.
- Las observaciones X(t) = [CO, NO2, PM2.5, O3, SO2, TVOC] son las mediciones del sensor.
- El modelo aprende de forma **no supervisada** que combinacion quimica corresponde a cada regimen.
- La prediccion a futuro se hace proyectando la distribucion de estados con la matriz de transicion.

### Â¿Por que HMM y no solo clustering?
El clustering (K-Means, GMM) asigna cada punto a un estado sin considerar la secuencia temporal.
El HMM incorpora explicitamente la dinamica: P(Z_t+1 | Z_t), la probabilidad de que el sistema
pase de un regimen a otro. Esto permite prediccion multi-step y captura la persistencia temporal
de los episodios de contaminacion.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from hmmlearn import hmm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10
})

# Constantes del proyecto
SENSORES_D = ['D_P1', 'D_P2']
OBSERVATION_FEATURES = ['CO_ppm', 'NO2_ppm', 'PM2_5_ugm3', 'O3_ppm', 'SO2_ppm', 'TVOC_ppb']

# Frecuencia aproximada de cada sensor (minutos por registro)
FREQ_MIN = {'D_P1': 1.02, 'D_P2': 1.57}

# Horizontes de prediccion en pasos (ceil(minutos / freq))
H_30 = {s: int(np.ceil(30 / f)) for s, f in FREQ_MIN.items()}  # D_P1: 30, D_P2: 20
H_60 = {s: int(np.ceil(60 / f)) for s, f in FREQ_MIN.items()}  # D_P1: 59, D_P2: 39

print("Horizontes de prediccion (pasos):")
for s in SENSORES_D:
    print(f"  {s}: t+30min â†’ {H_30[s]} pasos | t+60min â†’ {H_60[s]} pasos")

---
## 1. Carga de Datos

Los datos provienen de `Datos_maestro_sensores.xlsx`. Se filtra unicamente a los sensores D
(`D_P1` y `D_P2`), que son los sensores de referencia del proyecto con mediciones completas
de todos los contaminantes criterio.

**Â¿Por que leer el Excel y no los CSVs del ETL?**
El ETL anterior transformo los contaminantes a escala logaritmica y no guardo todas las columnas.
Para el HMM necesitamos los valores en **escala original** (ppm, Âµg/mÂ³) porque:
1. Las medias de emision Î¼_k deben ser interpretables quimicamente (ej. "CO = 0.8 ppm en trafico").
2. PM2.5, NO2 y O3 no se incluyeron como features en el ETL previo.
3. La normalizacion se aplica internamente con un StandardScaler, sin perder la referencia fisica.

In [ ]:
df_raw = pd.read_excel('Data/Datos_maestro_sensores.xlsx', parse_dates=['datetime'])

# Filtrar a sensores D
df = df_raw[df_raw['sensor_periodo'].isin(SENSORES_D)].copy()
df = df.sort_values(['sensor_periodo', 'datetime']).reset_index(drop=True)

print(f"Registros totales: {len(df)}")
print(f"\nRegistros por sensor:")
print(df['sensor_periodo'].value_counts())
print(f"\nColumnas disponibles: {list(df.columns)}")

In [ ]:
cols_necesarias = OBSERVATION_FEATURES + ['datetime', 'sensor_periodo', 'AQI']
faltantes = [c for c in cols_necesarias if c not in df.columns]
assert len(faltantes) == 0, f"Columnas faltantes: {faltantes}"

print("Verificacion OK â€” todas las columnas necesarias presentes")
print(f"\nRango temporal:")
for s in SENSORES_D:
    sub = df[df['sensor_periodo'] == s]
    print(f"  {s}: {sub['datetime'].min().date()} â†’ {sub['datetime'].max().date()} ({len(sub)} registros)")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

colores = {'D_P1': '#2196F3', 'D_P2': '#F44336'}

for i, feat in enumerate(OBSERVATION_FEATURES):
    for s in SENSORES_D:
        sub = df[df['sensor_periodo'] == s].set_index('datetime')
        axes[i].plot(sub.index, sub[feat], alpha=0.6, linewidth=0.5,
                     color=colores[s], label=s)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('Concentracion')
    axes[i].legend(fontsize=7)
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('Series temporales â€” Contaminantes criterio (Sensores D)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Data/HMM_01_series_contaminantes.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nEstadisticas descriptivas:")
display(df[OBSERVATION_FEATURES].describe().round(4))

---
## 2. Preprocesamiento

### 2.1 Segmentacion por gaps

El HMM **asume que los datos son una secuencia temporal continua**. Si hay una brecha
(por ejemplo, el sensor apagado 50 horas), el modelo interpretaria el salto como una
transicion de estado normal, lo que contamina la estimacion de la matriz de transicion A.

Solucion: identificar gaps > 5 minutos y cortar la serie en segmentos independientes.
Cada segmento es una secuencia continua. Al entrenar, se declaran sus longitudes por
separado en el parametro `lengths` de hmmlearn.

**Gaps por sensor:**
- D_P1: 0 gaps > 5 min â†’ completamente continuo (un solo segmento)
- D_P2: 4 gaps > 5 min, el mayor de ~50 horas (apagon 19-20 Nov)

In [ ]:
def segmentar_por_gaps(df_sensor, umbral_min=5):
    """
    Divide la serie de un sensor en segmentos continuos.
    Un gap > umbral_min minutos rompe la continuidad.
    Retorna el DataFrame con columna 'segment_id' anadida.
    """
    sub = df_sensor.copy().sort_values('datetime').reset_index(drop=True)
    diffs = sub['datetime'].diff().dt.total_seconds() / 60
    sub['gap_min'] = diffs.fillna(0)
    sub['gap_flag'] = sub['gap_min'] > umbral_min
    sub['segment_id'] = sub['gap_flag'].cumsum()
    return sub

# Aplicar por sensor
df_list = []
for s in SENSORES_D:
    sub = df[df['sensor_periodo'] == s].copy()
    sub = segmentar_por_gaps(sub, umbral_min=5)
    df_list.append(sub)

df = pd.concat(df_list).sort_values(['sensor_periodo', 'datetime']).reset_index(drop=True)

# Hacer que segment_id sea unico a traves de sensores
df['segment_global_id'] = df['sensor_periodo'] + '_seg' + df['segment_id'].astype(str)

print("Segmentos por sensor:")
for s in SENSORES_D:
    sub = df[df['sensor_periodo'] == s]
    n_segs = sub['segment_id'].nunique()
    print(f"  {s}: {n_segs} segmento(s)")
    for seg_id, grp in sub.groupby('segment_id'):
        print(f"    Seg {seg_id}: {grp['datetime'].min().date()} â†’ "
              f"{grp['datetime'].max().date()} ({len(grp)} registros)")

In [ ]:
n_nan = df[OBSERVATION_FEATURES].isna().sum()
print("NaN por feature:")
print(n_nan)

if n_nan.sum() > 0:
    print("\nImputando NaN con forward fill + backward fill por segmento...")
    def fill_segment(grp):
        grp[OBSERVATION_FEATURES] = grp[OBSERVATION_FEATURES].fillna(method='ffill').fillna(method='bfill')
        return grp
    df = df.groupby('segment_global_id', group_keys=False).apply(fill_segment)
    print(f"NaN restantes: {df[OBSERVATION_FEATURES].isna().sum().sum()}")
else:
    print("Sin NaN â€” OK")

### 2.2 Split temporal

El split se hace por fecha, no aleatoriamente, para simular prediccion real:
los datos anteriores al percentil 80 del tiempo son train, el resto es test.

**Importante:** El split respeta los segmentos â€” no se parte un segmento a la mitad
si no es necesario. En la practica, el percentil 80 cae dentro de D_P2 (que es el
sensor con mas datos y cubre hasta el 25 Nov).

In [ ]:
# Cutoff temporal global en el percentil 80 del rango de fechas
t_min = df['datetime'].min()
t_max = df['datetime'].max()
cutoff = t_min + (t_max - t_min) * 0.8

df_train = df[df['datetime'] <= cutoff].copy()
df_test  = df[df['datetime'] >  cutoff].copy()

print(f"Cutoff temporal: {cutoff.strftime('%Y-%m-%d %H:%M')}")
print(f"Train: {len(df_train)} registros ({df_train['datetime'].min().date()} â†’ {df_train['datetime'].max().date()})")
print(f"Test:  {len(df_test)} registros ({df_test['datetime'].min().date()} â†’ {df_test['datetime'].max().date()})")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train[OBSERVATION_FEATURES].values)
X_test_scaled  = scaler.transform(df_test[OBSERVATION_FEATURES].values)

print("Scaler ajustado sobre train.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

### 2.3 El parametro `lengths` en hmmlearn

`GaussianHMM.fit(X, lengths=[n1, n2, ...])` indica que X es en realidad
la concatenacion de n1+n2+... observaciones de secuencias independientes.
Sin `lengths`, hmmlearn asume que toda la secuencia es continua y calculara
probabilidades de transicion incorrectas entre segmentos.

In [ ]:
def calcular_lengths(df_split):
    """
    Calcula la longitud de cada segmento continuo dentro de un split.
    Retorna lista de enteros para el parametro lengths de hmmlearn.
    """
    lengths = []
    for seg_id, grp in df_split.groupby('segment_global_id', sort=False):
        lengths.append(len(grp))
    return lengths

lengths_train = calcular_lengths(df_train)
lengths_test  = calcular_lengths(df_test)

print(f"Segmentos en train: {len(lengths_train)} â†’ lengths: {lengths_train}")
print(f"Segmentos en test:  {len(lengths_test)} â†’ lengths: {lengths_test}")
assert sum(lengths_train) == len(df_train), "lengths_train no suma al total de filas"
assert sum(lengths_test)  == len(df_test),  "lengths_test no suma al total de filas"
print("Verificacion de lengths OK")

---
## 3. SelecciÃ³n del NÃºmero de Estados K

### Â¿CuÃ¡ntos regÃ­menes de calidad del aire existen?

El HMM requiere especificar K (nÃºmero de estados ocultos) a priori. Para elegirlo de forma
objetiva usamos el **Criterio de InformaciÃ³n Bayesiano (BIC)**:

$$BIC = -2 \cdot \log(L) + p \cdot \log(n)$$

Donde:
- $\log(L)$ es la log-verosimilitud del modelo ajustado (mayor = mejor ajuste)
- $p$ es el nÃºmero de parÃ¡metros libres (penalizaciÃ³n por complejidad)
- $n$ es el nÃºmero de observaciones

El BIC penaliza modelos con mÃ¡s parÃ¡metros para evitar sobreajuste. Se elige el K que
**minimiza** el BIC.

**ParÃ¡metros libres de GaussianHMM con covariance_type='full':**
- Matriz de transiciÃ³n A: KÃ—(Kâˆ’1) parÃ¡metros (cada fila suma 1)
- Medias de emisiÃ³n Î¼_k: K Ã— n_features
- Matrices de covarianza Î£_k (full): K Ã— n_featuresÂ² parÃ¡metros

Se prueban K = 2, 3, 4, 5, 6, 7, 8.

In [ ]:
def calcular_bic(X, lengths, k):
    """
    Entrena un GaussianHMM con k estados y calcula su BIC sobre X.
    Retorna (bic, modelo_entrenado).
    """
    modelo = hmm.GaussianHMM(
        n_components=k,
        covariance_type='full',
        n_iter=100,
        random_state=42,
        verbose=False
    )
    modelo.fit(X, lengths=lengths)

    log_likelihood = modelo.score(X, lengths=lengths)
    n_features = X.shape[1]

    # ParÃ¡metros libres
    n_params = (k * (k - 1) +           # matriz de transiciÃ³n (sin diagonal libre: k filas, k-1 libres cada una)
                k * n_features +         # medias
                k * n_features ** 2)     # covarianzas full

    bic = -2 * log_likelihood + n_params * np.log(len(X))
    return bic, modelo

In [ ]:
print("Calculando BIC para K = 2..8 (puede tardar ~2-3 min)...")
resultados_bic = {}

for k in range(2, 9):
    bic, modelo_k = calcular_bic(X_train_scaled, lengths_train, k)
    resultados_bic[k] = {'bic': bic, 'modelo': modelo_k}
    print(f"  K={k} â†’ BIC={bic:.2f}")

K_optimo = min(resultados_bic, key=lambda k: resultados_bic[k]['bic'])
print(f"
K Ã³ptimo (BIC mÃ­nimo): K={K_optimo}")

In [ ]:
ks   = list(resultados_bic.keys())
bics = [resultados_bic[k]['bic'] for k in ks]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, bics, marker='o', color='#2196F3', linewidth=2, markersize=7)
ax.axvline(K_optimo, color='#F44336', linestyle='--', linewidth=1.2,
           label=f'K Ã³ptimo = {K_optimo}')
ax.scatter([K_optimo], [resultados_bic[K_optimo]['bic']],
           color='#F44336', s=100, zorder=5)
ax.set_xlabel('NÃºmero de estados K')
ax.set_ylabel('BIC')
ax.set_title('SelecciÃ³n de K â€” Criterio BIC (menor = mejor)', fontweight='bold')
ax.legend()
ax.set_xticks(ks)

plt.tight_layout()
plt.savefig('Data/HMM_02_bic_vs_k.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Entrenamiento del GaussianHMM Final

Con K={K_optimo} estados seleccionado por BIC, se usa el modelo ya entrenado en el loop anterior
(no es necesario re-entrenar). Se reportan los parámetros aprendidos:

- **π** — distribución de estado inicial
- **A** — matriz de transición K×K
- **μ_k** — media de observaciones en estado k (en espacio normalizado y original)

In [ ]:
model = resultados_bic[K_optimo]['modelo']

print(f"=== GaussianHMM con K={K_optimo} estados ===")
print(f"\nDistribución inicial π:")
for k in range(K_optimo):
    print(f"  Estado {k+1}: {model.startprob_[k]*100:.1f}%")

print(f"\nMatriz de transición A (filas = estado actual, cols = estado siguiente):")
A_df = pd.DataFrame(
    model.transmat_.round(3),
    index=[f'Z={k+1}' for k in range(K_optimo)],
    columns=[f'→Z={k+1}' for k in range(K_optimo)]
)
display(A_df)

print(f"\nLog-likelihood en train: {model.score(X_train_scaled, lengths=lengths_train):.2f}")

In [ ]:
# Desnormalizar las medias para interpretación química
means_original = scaler.inverse_transform(model.means_)

df_medias = pd.DataFrame(
    means_original,
    columns=OBSERVATION_FEATURES,
    index=[f'Estado {k+1}' for k in range(K_optimo)]
)

print("\nMedias de emisión por estado (escala original):")
display(df_medias.round(4))
print("\nInterpretación química esperada:")
print("  CO alto + NO2 alto + TVOC alto → régimen de tráfico")
print("  O3 alto + NO2 bajo             → régimen fotoquímico")
print("  Todos bajos                    → aire limpio")
print("  PM2.5 alto + CO moderado       → episodio mixto/quemas")

---
## 5. Decodificación de Estados (Algoritmo de Viterbi)

El algoritmo de Viterbi encuentra la secuencia de estados ocultos más probable dada
la secuencia de observaciones: argmax P(Z_0:T | X_0:T).

También se calculan las **probabilidades posteriores** P(Z_t = k | X_0:t) con el
algoritmo forward, que es la entrada para la predicción multi-step.

In [ ]:
# Decodificación por Viterbi
estados_test = model.predict(X_test_scaled, lengths=lengths_test)

# Probabilidades posteriores (algoritmo forward)
log_probs, posteriors_test = model.score_samples(X_test_scaled, lengths=lengths_test)

print(f"Estados decodificados: {len(estados_test)} timesteps")
print(f"\nDistribución de estados en test:")
for k in range(K_optimo):
    pct = (estados_test == k).mean() * 100
    print(f"  Estado {k+1}: {pct:.1f}% del tiempo")

In [ ]:
# Usar el primer segmento de test para la visualización
idx_inicio = 0
primer_seg_len = lengths_test[0]

fechas_seg = df_test['datetime'].iloc[:primer_seg_len].values
estados_seg = estados_test[:primer_seg_len]
co_seg = df_test['CO_ppm'].iloc[:primer_seg_len].values
pm25_seg = df_test['PM2_5_ugm3'].iloc[:primer_seg_len].values

# Paleta de colores por estado
PALETA = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#00BCD4', '#795548', '#607D8B']
colores_estado = {k: PALETA[k] for k in range(K_optimo)}

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Panel 1: Estado oculto como banda de color
ax = axes[0]
for t in range(len(estados_seg) - 1):
    ax.axvspan(t, t + 1, color=colores_estado[estados_seg[t]], alpha=0.6)
ax.set_xlim(0, len(estados_seg))
ax.set_ylabel('Estado')
ax.set_yticks([])
patches = [mpatches.Patch(color=colores_estado[k], label=f'Estado {k+1}')
           for k in range(K_optimo)]
ax.legend(handles=patches, loc='upper right', fontsize=7, ncol=K_optimo)
ax.set_title('Secuencia de estados ocultos — primer segmento de test', fontweight='bold')

# Panel 2: CO_ppm
axes[1].plot(range(len(co_seg)), co_seg, linewidth=0.7, color='#37474F')
axes[1].set_ylabel('CO (ppm)')
for t in range(len(estados_seg) - 1):
    axes[1].axvspan(t, t + 1, color=colores_estado[estados_seg[t]], alpha=0.15)

# Panel 3: PM2.5
axes[2].plot(range(len(pm25_seg)), pm25_seg, linewidth=0.7, color='#37474F')
axes[2].set_ylabel('PM2.5 (µg/m³)')
axes[2].set_xlabel('Timestep')
for t in range(len(estados_seg) - 1):
    axes[2].axvspan(t, t + 1, color=colores_estado[estados_seg[t]], alpha=0.15)

plt.tight_layout()
plt.savefig('Data/HMM_03_estados_temporales.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

im = ax.imshow(df_medias.values, aspect='auto', cmap='RdYlGn_r')
ax.set_xticks(range(len(OBSERVATION_FEATURES)))
ax.set_xticklabels(OBSERVATION_FEATURES, rotation=30, ha='right', fontsize=9)
ax.set_yticks(range(K_optimo))
ax.set_yticklabels([f'Estado {k+1}' for k in range(K_optimo)])
ax.set_title('Medias de emisión por estado (escala original)', fontweight='bold')

for i in range(K_optimo):
    for j in range(len(OBSERVATION_FEATURES)):
        ax.text(j, i, f'{df_medias.values[i, j]:.3f}',
                ha='center', va='center', fontsize=7, color='black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('Data/HMM_04_medias_estados.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Tomar el posterior del último timestep del primer segmento de test
posterior_ultimo = posteriors_test[primer_seg_len - 1]

# Identificar a qué sensor pertenece el primer segmento de test
sensor_primer_seg = df_test['sensor_periodo'].iloc[0]
h30 = H_30[sensor_primer_seg]
h60 = H_60[sensor_primer_seg]

print(f"Sensor del primer segmento de test: {sensor_primer_seg}")
print(f"H para t+30min: {h30} pasos | H para t+60min: {h60} pasos")

# Predicción t+30
dist_30 = predecir_estado_futuro(model, posterior_ultimo, H=h30)
conc_30 = predecir_concentracion(model, dist_30, scaler)

# Predicción t+60
dist_60 = predecir_estado_futuro(model, posterior_ultimo, H=h60)
conc_60 = predecir_concentracion(model, dist_60, scaler)

print(f"\n--- Distribución de estados ---")
print(f"{'Estado':<10} {'Actual (t)':<15} {'t+30min':<15} {'t+60min':<15}")
for k in range(K_optimo):
    print(f"Estado {k+1:<4} {posterior_ultimo[k]*100:>10.1f}%  "
          f"{dist_30[k]*100:>10.1f}%  {dist_60[k]*100:>10.1f}%")

print(f"\n--- Concentración esperada ---")
print(f"{'Feature':<15} {'t+30min':>12} {'t+60min':>12}")
for nombre, v30, v60 in zip(OBSERVATION_FEATURES, conc_30, conc_60):
    print(f"{nombre:<15} {v30:>12.4f} {v60:>12.4f}")

In [ ]:
def predecir_estado_futuro(model, posterior_t, H):
    """
    Proyecta la distribución de estado H pasos hacia adelante.
    
    posterior_t: array (K,) — P(Z_t = k | X_0:t)
    H: número de pasos hacia adelante
    Retorna: array (K,) — P(Z_{t+H} = k | X_0:t)
    """
    A = model.transmat_
    dist = posterior_t.copy()
    for _ in range(H):
        dist = dist @ A
    return dist


def predecir_concentracion(model, dist_futura, scaler):
    """
    Calcula la concentración esperada como suma ponderada de medias de emisión.
    
    dist_futura: array (K,) — distribución de estado en t+H
    Retorna: array (n_features,) en escala original
    """
    # Expectativa: E[X] = sum_k P(Z=k) * mu_k
    conc_norm = np.dot(dist_futura, model.means_)  # (n_features,)
    conc = scaler.inverse_transform(conc_norm.reshape(1, -1)).flatten()
    return conc

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
estados_labels = [f'Estado {k+1}' for k in range(K_optimo)]
x = np.arange(K_optimo)

for ax, (dist, titulo) in zip(axes, [
    (posterior_ultimo, 'Actual (t)'),
    (dist_30, f't+30 min ({h30} pasos)'),
    (dist_60, f't+60 min ({h60} pasos)')
]):
    bars = ax.bar(x, dist * 100, color=[PALETA[k] for k in range(K_optimo)],
                  edgecolor='white', width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(estados_labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Probabilidad (%)')
    ax.set_ylim(0, 105)
    ax.set_title(titulo, fontweight='bold')
    for bar, v in zip(bars, dist):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{v*100:.0f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle('Distribución de estados predicha en horizontes temporales', fontweight='bold')
plt.tight_layout()
plt.savefig('Data/HMM_05_pred_distribucion.png', dpi=120, bbox_inches='tight')
plt.show()